# FactSet Revere (sample) supply-chain extraction

Pulls real SUPPLIER/CUSTOMER relationships from `factsamp_revere.wrds_relationship`
(the sample/trial FactSet Revere schema this WRDS account has access to -- the
exact data source the paper uses, `FactSet Revere - Supply Chain / Relationships`,
just the sample/trial tier rather than full production).

Confirmed via manual probing (2026-09-11):
- `factsamp_revere.wrds_relationship`: 27,026 total rows, 10,791 of which are
  `rel_type in ('SUPPLIER','CUSTOMER')` -- the rest are COMPETITOR/PARTNER-* noise.
- SUPPLIER/CUSTOMER subset: 2,116 distinct source companies, 1,126 distinct target
  companies, dated 2003-04-03 through 2015-05-11 (`start_`/`end_` columns).
- `source_cusip`/`target_cusip` non-null for ~8,476/9,575 of the 10,791 rows
  (~79%/89%) -- the rest have only a ticker or neither.
- `wrdsapps_link_crsp_factset.fscrsplink` (343,249 rows) has a direct `cusip` ->
  `permno` column, which chains into this project's existing `permno` -> `gvkey`
  crosswalk (`data/ccm_link.parquet`), giving a clean path to our gvkey system
  without fuzzy name matching (unlike the earlier SEC-filings knowledge-graph
  attempt, which was bottlenecked at 17% name-match rate).

**Cell 1 below is run by the user** (interactive WRDS password prompt / `.pgpass`).
Every cell after that reuses the `db` connection object it creates.

In [1]:
# SQLAlchemy 2.x compatibility patch: the installed wrds package (3.1.x)
# predates SQLAlchemy 2.x's stricter Connection.execute() API, which requires
# raw SQL strings to be wrapped in sqlalchemy.text(...) rather than passed
# bare. No SQLAlchemy 1.4.x wheel exists for this Python version, so this
# monkeypatches Connection.execute to auto-wrap bare strings before
# delegating to the real method.
#
# Guarded with a marker attribute so re-running this cell (e.g. after a
# kernel restart, or just re-executing cells out of order) doesn't wrap an
# already-patched execute() a second time, which caused infinite recursion.
import sqlalchemy as sa
from sqlalchemy.engine import Connection

if not getattr(Connection.execute, "_is_sa2_compat_patch", False):
    _original_execute = Connection.execute

    def _execute_compat(self, statement, *args, **kwargs):
        if isinstance(statement, str):
            statement = sa.text(statement)
        return _original_execute(self, statement, *args, **kwargs)

    _execute_compat._is_sa2_compat_patch = True
    Connection.execute = _execute_compat

import wrds

# Run this cell yourself -- it will prompt for your WRDS username/password
# (or use the .pgpass file already saved from earlier).
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
Loading library list...
Done


## 1. Pull the raw SUPPLIER/CUSTOMER relationships

Only `rel_type in ('SUPPLIER','CUSTOMER')` -- excludes COMPETITOR and the
PARTNER-* subtypes, which are not supply-chain links.

In [2]:
import pandas as pd

TABLE = "factsamp_revere.wrds_relationship"

raw = db.raw_sql(f"""
    select start_, end_, rel_type, source_company_id, target_company_id,
           source_name, source_ticker, source_cusip,
           target_name, target_ticker, target_cusip
    from {TABLE}
    where rel_type in ('SUPPLIER', 'CUSTOMER')
""")

print(f"raw rows: {len(raw):,}")
print(f"date range: {raw['start_'].min()} to {raw['start_'].max()}")
raw.head()

raw rows: 10,791
date range: 2003-04-03 to 2015-05-11


,start_,end_,rel_type,source_company_id,target_company_id,source_name,source_ticker,source_cusip,target_name,target_ticker,target_cusip
0,2004-05-12,2005-01-26,CUSTOMER,10805,2785,Verisign Inc,VRSN,92343E102,NaN,NaN,NaN
1,2005-01-26,2005-01-26,CUSTOMER,10805,2785,Verisign Inc,VRSN,92343E102,NaN,NaN,NaN
2,2011-03-24,2011-03-24,CUSTOMER,10805,1518552,Verisign Inc,VRSN,92343E102,NaN,NaN,NaN
3,2003-06-08,2004-04-14,CUSTOMER,897,3078,NaN,NaN,NaN,Dell Inc,DELL,24702R101
4,2004-04-14,2005-04-20,CUSTOMER,897,3078,NaN,NaN,NaN,Dell Inc,DELL,24702R101


## 2. Resolve CUSIP -> gvkey via fscrsplink + ccm_link

`fscrsplink` maps `cusip` -> `permno` directly (FactSet's own CRSP link table).
`data/ccm_link.parquet` (already used throughout this project's pipeline) maps
`permno` -> `gvkey`. Chaining the two avoids any fuzzy name matching.

In [3]:
cusip_link = db.raw_sql("""
    select cusip, permno, link_bdate, link_edate
    from wrdsapps_link_crsp_factset.fscrsplink
    where cusip is not null and permno is not null
""")
cusip_link = cusip_link.drop_duplicates(subset=['cusip', 'permno'])
print(f"cusip -> permno link rows: {len(cusip_link):,}, distinct cusips: {cusip_link['cusip'].nunique():,}")

# Take the earliest-linked permno per cusip if a cusip maps to more than one
# (rare, but the link table isn't guaranteed 1:1).
cusip_to_permno = (
    cusip_link.sort_values('link_bdate')
    .drop_duplicates(subset=['cusip'], keep='first')
    .set_index('cusip')['permno']
)
print(f"unique cusip -> permno mapping: {len(cusip_to_permno):,} cusips")

cusip -> permno link rows: 53,645, distinct cusips: 53,645
unique cusip -> permno mapping: 53,645 cusips


In [4]:
ccm = pd.read_parquet('../data/ccm_link.parquet')
ccm = ccm[ccm['linktype'].isin(['LU', 'LC']) & ccm['linkprim'].isin(['P', 'C'])].copy()
ccm['permno'] = ccm['permno'].astype(int)

# Keep the most recent gvkey per permno if there's more than one link row
# (matches the same simplification used in build_supply_chain_features.py).
permno_to_gvkey = (
    ccm.sort_values('linkdt')
    .drop_duplicates(subset=['permno'], keep='last')
    .set_index('permno')['gvkey']
)
print(f"unique permno -> gvkey mapping: {len(permno_to_gvkey):,} permnos")

unique permno -> gvkey mapping: 29,567 permnos


## 3. Map source/target CUSIPs to gvkeys, build directed edges

`SUPPLIER`: `source` supplies `target` -> `source` = supplier, `target` = customer.
`CUSTOMER`: `source` has `target` as its customer -> same direction, `target` = customer.
(Revere's own schema already encodes direction via which side of the relationship
`source`/`target` occupy -- both rel_types here describe the same source->target
supply direction, just recorded from a different reporting company's perspective.)

In [5]:
def cusip_to_gvkey(cusip_series):
    # fscrsplink.cusip is 8-char (header/issuer-level CUSIP, no check digit);
    # Revere's source_cusip/target_cusip are 9-char (full security-level
    # CUSIP, with trailing check digit) -- confirmed via the diagnostic cell
    # above (cusip_link['cusip'] is uniformly length 8, raw['source_cusip']
    # is uniformly length 9). Truncate to the first 8 chars before mapping.
    cusip8 = cusip_series.str[:8]
    permno = cusip8.map(cusip_to_permno)
    gvkey = permno.map(permno_to_gvkey)
    return gvkey

edges = raw.copy()
edges['supplier_gvkey'] = cusip_to_gvkey(edges['source_cusip'])
edges['customer_gvkey'] = cusip_to_gvkey(edges['target_cusip'])

matched = edges.dropna(subset=['supplier_gvkey', 'customer_gvkey']).copy()
print(f"raw SUPPLIER/CUSTOMER rows: {len(edges):,}")
print(f"rows with BOTH sides resolved to gvkey: {len(matched):,}  ({len(matched)/len(edges)*100:.1f}%)")
print(f"distinct gvkeys touched: {len(set(matched['supplier_gvkey']) | set(matched['customer_gvkey'])):,}")

matched[['start_', 'end_', 'rel_type', 'source_name', 'supplier_gvkey', 'target_name', 'customer_gvkey']].head(15)

raw SUPPLIER/CUSTOMER rows: 10,791
rows with BOTH sides resolved to gvkey: 5,058  (46.9%)
distinct gvkeys touched: 1,085


,start_,end_,rel_type,source_name,supplier_gvkey,target_name,customer_gvkey
8,2003-04-03,2003-06-08,CUSTOMER,Atmel Corp,023767,Microsoft Corp,012141
9,2003-06-08,2003-06-08,CUSTOMER,Atmel Corp,023767,Microsoft Corp,012141
10,2006-04-21,2007-06-13,CUSTOMER,Tellabs Inc,010420,Microsoft Corp,012141
11,2007-06-13,2007-06-13,CUSTOMER,Tellabs Inc,010420,Microsoft Corp,012141
12,2003-07-27,2004-06-16,CUSTOMER,Teltronics Inc,016405,International Business Machines Corp IBM,006066
13,2004-06-16,2005-04-27,CUSTOMER,Teltronics Inc,016405,International Business Machines Corp IBM,006066
14,2005-04-27,2005-06-15,CUSTOMER,Teltronics Inc,016405,International Business Machines Corp IBM,006066
15,2005-06-15,2005-06-15,CUSTOMER,Teltronics Inc,016405,International Business Machines Corp IBM,006066
16,2007-05-23,2008-06-11,CUSTOMER,Teradyne Inc,010453,Dell Inc,014489
17,2008-06-11,2008-06-11,CUSTOMER,Teradyne Inc,010453,Dell Inc,014489


In [6]:
# DIAGNOSTIC: check CUSIP format/length on both sides of the join -- the
# 0% match rate strongly suggests a length mismatch (9-char security CUSIP
# vs 8-char header CUSIP), same class of bug as the earlier gvkey
# zero-padding issue this session.
print("cusip_link['cusip'] sample values and lengths:")
print(cusip_link['cusip'].head(10).tolist())
print("cusip_link['cusip'] length distribution:")
print(cusip_link['cusip'].str.len().value_counts())

print()
print("raw['source_cusip'] sample values and lengths:")
print(raw['source_cusip'].dropna().head(10).tolist())
print("raw['source_cusip'] length distribution:")
print(raw['source_cusip'].dropna().str.len().value_counts())


cusip_link['cusip'] sample values and lengths:
['68391610', '39040610', '29274A10', '29274A20', '29269V10', '36720410', '60740110', '83623410', '05978R10', '39031810']
cusip_link['cusip'] length distribution:
cusip
8    53645
Name: count, dtype: int64

raw['source_cusip'] sample values and lengths:
['92343E102', '92343E102', '92343E102', '037833100', '049513104', '049513104', '879664100', '879664100', '879698306', '879698306']
raw['source_cusip'] length distribution:
source_cusip
9    8476
Name: count, dtype: int64


## 4. Compare against the existing WRDS Compustat Segment graph

Sanity check before merging: how much does this add on top of what
`data/scg_edges_yearly.parquet` (the current pipeline's supply-chain source)
already covers?

In [7]:
existing = pd.read_parquet('../data/scg_edges_yearly.parquet')
print("existing scg_edges_yearly.parquet columns:", list(existing.columns))

existing_all_gvkeys = set(existing['supplier_gvkey'].astype(str)) if 'supplier_gvkey' in existing.columns else set()
revere_gvkeys = set(matched['supplier_gvkey'].astype(str)) | set(matched['customer_gvkey'].astype(str))

new_gvkeys = revere_gvkeys - existing_all_gvkeys
print(f"gvkeys in FactSet Revere sample: {len(revere_gvkeys):,}")
print(f"gvkeys already in WRDS Compustat Segment graph: {len(existing_all_gvkeys):,}")
print(f"NEW gvkeys FactSet Revere would add: {len(new_gvkeys):,}")

existing scg_edges_yearly.parquet columns: ['year', 'supplier_permno', 'customer_permno', 'supplier_gvkey', 'customer_name', 'sales_to_customer']
gvkeys in FactSet Revere sample: 1,085
gvkeys already in WRDS Compustat Segment graph: 2,916
NEW gvkeys FactSet Revere would add: 621


## 5. Build the yearly edge table (same schema as scg_edges_yearly.parquet)

Uses `start_` year as the disclosure year, matching the existing pipeline's
`year`/`supplier_gvkey`/`customer_gvkey` convention so this can be fed straight
into `build_supply_chain_features.py`'s month-expansion logic (or a copy of it).

In [8]:
revere_edges_yearly = matched.copy()
revere_edges_yearly['year'] = pd.to_datetime(revere_edges_yearly['start_']).dt.year
revere_edges_yearly = revere_edges_yearly[[
    'year', 'supplier_gvkey', 'customer_gvkey', 'rel_type', 'source_name', 'target_name'
]].drop_duplicates(subset=['year', 'supplier_gvkey', 'customer_gvkey'])

revere_edges_yearly['supplier_gvkey'] = revere_edges_yearly['supplier_gvkey'].astype(str).str.zfill(6)
revere_edges_yearly['customer_gvkey'] = revere_edges_yearly['customer_gvkey'].astype(str).str.zfill(6)

print(f"final yearly edge rows: {len(revere_edges_yearly):,}")
print(f"year range: {revere_edges_yearly['year'].min()} - {revere_edges_yearly['year'].max()}")
revere_edges_yearly.to_parquet('../data/scg_edges_factset_revere_sample.parquet', index=False)
print("wrote ../data/scg_edges_factset_revere_sample.parquet")
revere_edges_yearly.head(10)

final yearly edge rows: 4,435
year range: 2003 - 2015
wrote ../data/scg_edges_factset_revere_sample.parquet


,year,supplier_gvkey,customer_gvkey,rel_type,source_name,target_name
8,2003,023767,012141,CUSTOMER,Atmel Corp,Microsoft Corp
10,2006,010420,012141,CUSTOMER,Tellabs Inc,Microsoft Corp
11,2007,010420,012141,CUSTOMER,Tellabs Inc,Microsoft Corp
12,2003,016405,006066,CUSTOMER,Teltronics Inc,International Business Machines Corp IBM
13,2004,016405,006066,CUSTOMER,Teltronics Inc,International Business Machines Corp IBM
14,2005,016405,006066,CUSTOMER,Teltronics Inc,International Business Machines Corp IBM
16,2007,010453,014489,CUSTOMER,Teradyne Inc,Dell Inc
17,2008,010453,014489,CUSTOMER,Teradyne Inc,Dell Inc
18,2007,010453,006066,CUSTOMER,Teradyne Inc,International Business Machines Corp IBM
19,2008,010453,006066,CUSTOMER,Teradyne Inc,International Business Machines Corp IBM


In [10]:
# Year-by-year breakdown of the final matched edge table
print("Final yearly edge rows by year:")
print(revere_edges_yearly['year'].value_counts().sort_index())
print()
print("Distinct gvkeys touched per year (source or target):")
by_year_gvkeys = revere_edges_yearly.groupby('year').apply(
    lambda g: len(set(g['supplier_gvkey']) | set(g['customer_gvkey'])), include_groups=False
)
print(by_year_gvkeys)


Final yearly edge rows by year:
year
2003    704
2004    374
2005    386
2006    320
2007    230
2008    212
2009    149
2010    236
2011    386
2012    457
2013    326
2014    382
2015    273
Name: count, dtype: int64

Distinct gvkeys touched per year (source or target):
year
2003    461
2004    302
2005    325
2006    263
2007    207
2008    184
2009    140
2010    208
2011    294
2012    351
2013    284
2014    310
2015    228
dtype: int64


## 6. Close the connection

In [9]:
db.close()